### Script to replace the IPI and Length from original packets in a PCAP

In [1]:
# Later, load the model
from keras.models import load_model
import joblib
import struct
from scapy.all import rdpcap, wrpcap, Raw, IP, TCP, UDP, RTP, PcapNgWriter
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, PowerTransformer
import sys
from tqdm import tqdm  # Import tqdm for the progress bar
from collections import deque
import tensorflow as tf
import pandas as pd

2025-06-06 15:14:22.424485: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749215662.437004   12411 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749215662.440686   12411 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-06 15:14:22.453472: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


### PCAP/CSV mode
Switch between PCAP input files or CSV files \
MODE = 4 (CSV - two files)

In [2]:
MODE=4

### (Optional) Force to use only CPU

In [3]:
tf.config.set_visible_devices([], 'GPU')

## Load models

### Load first model (CG only)

In [4]:
model1_path='../../../FlowSync/Cloud Gaming/DS_CG_IPI/UL/model10_UL.keras'
loaded_model1 = load_model(model1_path)
print("Model loaded successfully.")
#loaded_model1.summary()

Model loaded successfully.


In [5]:
# model2_path='../../../FlowSync/Cloud Gaming/DS_CG_IPI/UL/model10_UL.keras'
# loaded_model2 = load_model(model2_path)
# print("Model loaded successfully.")
# #loaded_model1.summary()

### Load data shape

In [6]:
import numpy as np
import json

with np.load(model1_path) as npz_file:
    for array_name in npz_file.files:
        print(f"Loading array '{array_name}'")
        
        if 'config.json' in array_name:
            # Decode bytes to string
            json_bytes = npz_file[array_name]
            json_str = json_bytes.decode('utf-8')
            
            # Load JSON string into a Python dictionary
            data = json.loads(json_str)
            print(f"JSON data: {data}")

            input_shape1 = data['config']['layers'][1]['build_config']['input_shape']
            print(f"Data shape: {input_shape1}")

Loading array 'metadata.json'
Loading array 'config.json'
JSON data: {'module': 'keras.src.models.functional', 'class_name': 'Functional', 'config': {'name': 'functional', 'trainable': True, 'layers': [{'module': 'keras.layers', 'class_name': 'InputLayer', 'config': {'batch_shape': [None, 10, 1], 'dtype': 'float32', 'sparse': False, 'name': 'input_layer'}, 'registered_name': None, 'name': 'input_layer', 'inbound_nodes': []}, {'module': 'keras.layers', 'class_name': 'Conv1D', 'config': {'name': 'conv1d', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None}, 'filters': 128, 'kernel_size': [3], 'strides': [1], 'padding': 'same', 'data_format': 'channels_last', 'dilation_rate': [1], 'groups': 1, 'activation': 'tanh', 'use_bias': True, 'kernel_initializer': {'module': 'keras.initializers', 'class_name': 'GlorotUniform', 'config': {'seed': None}, 'registered_name': None}, 'bias_initializer': {'module': 'keras.ini

In [7]:
input_shape1

[None, 10, 1]

### Define constants. We assume input_shape1 is the same as input_shape2. Otherwise, it `won't` work!!

In [8]:

WINDOW_SIZE = input_shape1[1]  # Size of the sliding window
MODIFY_PERCENT = 100  # Percentage of packets to modify (e.g., 10%, 20%, 30%, etc.)
MODIFY_COUNT = round((MODIFY_PERCENT / 100) * WINDOW_SIZE)  # Adjust count dynamically based on window size
NUM_FEATURES = input_shape1[2] # number of features that the model was trained on

print(f"WINDOW_SIZE: {WINDOW_SIZE}")
print(f"NUM_FEATURES: {NUM_FEATURES}")
print(f"MODIFY_COUNT: {MODIFY_COUNT}")

WINDOW_SIZE: 10
NUM_FEATURES: 1
MODIFY_COUNT: 10


### (Model 1 - CG only) Load or Fit scaler parameters

In [9]:
def fit_scaler_custom(data, scaler_path, power_transformer_path, apply_yeo_johnson=False, load=False):
    if load == False:
        if apply_yeo_johnson:
            power_transformer_path = PowerTransformer(method='yeo-johnson', standardize=False)
            data = pd.DataFrame(power_transformer_path.fit_transform(data), columns=data.columns)
            print("data2:", data.head())
            scaler = MinMaxScaler(feature_range=(-1,1))
            scaler.fit(data)
            print("scaler2:", scaler)
            return scaler, power_transformer_path
        else:
            scaler = MinMaxScaler(feature_range=(-1, 1))
            scaler.fit(data)
            print("data1:", data.head())
            print("scaler1:", scaler)
            return scaler, _
    else:
        if apply_yeo_johnson:
            scaler = joblib.load(scaler_path)
            power_transformer_path = joblib.load(power_transformer_path)
            return scaler, power_transformer_path
        else:
            scaler = joblib.load(scaler_path)
            return scaler, _

### Set the percentage of modified packets w/ a sliding window a prediction over original data only 

In [10]:
# Function to generate fixed predictions (stub for demonstration purposes)
def generate_fixed_prediction():
    predictions = np.array([[1, 10000]], dtype=float)  # Dummy [IPI, PS] prediction for demonstration purposes
    print(predictions.shape)
    return np.array([[1, 10000]], dtype=float)  # Dummy [IPI, PS] prediction for demonstration purposes

In [11]:
# Function to generate random integer predictions within a specified range (integer values)
def generate_rand_prediction_uniform(min_ipi, max_ipi, min_ps, max_ps):
    # Generate random integer values within the given ranges
    random_ipi = np.random.randint(min_ipi, max_ipi + 1)  # Inclusive range for IPI
    random_ps = np.random.randint(min_ps, max_ps + 1)    # Inclusive range for PS
    predictions = np.array([[random_ipi, random_ps]], dtype=int) # integer values
    return predictions

In [12]:
import numpy as np

def generate_rand_predictions_gaussian(mean_ipi, stddev_ipi, mean_ps, stddev_ps):
    """
    Generate random integer predictions based on a Gaussian (normal) distribution.
    
    Args:
        mean_ipi (float): Mean of the IPI (inter-packet interval) distribution.
        stddev_ipi (float): Standard deviation of the IPI distribution.
        mean_ps (float): Mean of the PS (packet size) distribution.
        stddev_ps (float): Standard deviation of the PS distribution.
    
    Returns:
        np.ndarray: A 2D array with shape (1, 2) containing the IPI and PS predictions as integers.
    """
    random_ipi = max(0, int(np.random.normal(mean_ipi, stddev_ipi)))  # Ensure non-negative values
    random_ps = max(0, int(np.random.normal(mean_ps, stddev_ps)))    # Ensure non-negative values
    predictions = np.array([[random_ipi, random_ps]], dtype=int)
    return predictions


In [13]:
def generate_model_prediction(window_packets, scaler, power_transformer, loaded_model, input_shape, apply_yeo_johnson=False):
    # Stack the input tensors along a new axis to form a single batch
    window = np.stack(window_packets, axis=0)
    
    # Reshape to add the batch dimension
    window = np.expand_dims(window, axis=0)
    
    # Call the model directly instead of predict
    predictions = loaded_model(window, training=False)
    
    # If predictions have unexpected dimensions, provide detailed debugging information
    if len(predictions.shape) != input_shape[-1]: # e.g., (1, 2) = (1 sample, 2 features)
        raise ValueError(f"Unexpected predictions shape: {predictions.shape}. Expected 2D tensor (batch_size, num_features).")

    # Add dummy columns to match the scaler's expected input shape
    num_predictions = predictions.shape[0]
    if predictions.shape[1] > NUM_FEATURES:
        raise ValueError(f"Predictions have more features ({predictions.shape[1]}) than expected NUM_FEATURES ({NUM_FEATURES}).")
    
    if NUM_FEATURES != predictions.shape[1]:
        dummy_columns = np.random.rand(num_predictions, NUM_FEATURES - predictions.shape[1])
        predictions_with_dummy = np.hstack([predictions, dummy_columns])
        predictions_original_scale = scaler.inverse_transform(predictions_with_dummy)
        predictions_original_scale = predictions_original_scale[:, :predictions.shape[1]]

        if apply_yeo_johnson:
            predictions_original_scale = power_transformer.inverse_transform(predictions_original_scale)
    else:
        predictions_original_scale = scaler.inverse_transform(predictions)
        predictions_original_scale = predictions_original_scale[:, :predictions.shape[1]]
        
        if apply_yeo_johnson:
            predictions_original_scale = power_transformer.inverse_transform(predictions_original_scale)

    
    return predictions_original_scale

In [14]:
def generate_model_prediction_v2(window_packets, scaler, power_transformer, loaded_model, input_shape, apply_yeo_johnson=False):
    # Stack the input tensors along a new axis to form a single batch
    window = np.expand_dims(np.stack(window_packets, axis=0), axis=0)

    # Call the model to get predictions
    predictions = loaded_model(window, training=False)

    # Validate the shape of predictions
    if predictions.shape[1] != input_shape[-1]:  # Compare the number of features
        raise ValueError(f"Unexpected predictions shape: {predictions.shape}. Expected (batch_size, {input_shape[-1]}).")

    # Process predictions for output
    num_predictions = predictions.shape[0]
    if predictions.shape[1] > NUM_FEATURES:
        raise ValueError(f"Predictions have more features ({predictions.shape[1]}) than expected NUM_FEATURES ({NUM_FEATURES}).")

    if NUM_FEATURES != predictions.shape[1]:
        dummy_columns = np.random.rand(num_predictions, NUM_FEATURES - predictions.shape[1])
        predictions_with_dummy = np.hstack([predictions, dummy_columns])
        predictions_original_scale = scaler.inverse_transform(predictions_with_dummy)
        predictions_original_scale = predictions_original_scale[:, :predictions.shape[1]]

        if apply_yeo_johnson:
            predictions_original_scale = power_transformer.inverse_transform(predictions_original_scale)
    else:
        predictions_original_scale = scaler.inverse_transform(predictions)
        predictions_original_scale = predictions_original_scale[:, :predictions.shape[1]]
        
        if apply_yeo_johnson:
            predictions_original_scale = power_transformer.inverse_transform(predictions_original_scale)

    return predictions_original_scale


In [15]:
def generate_model_prediction_v1(window_packets, scaler, loaded_model, input_shape):
    # Stack the input tensors along a new axis to form a single batch
    window = np.expand_dims(np.stack(window_packets, axis=0), axis=0)

    # Call the model to get predictions
    predictions = loaded_model(window, training=False)

    # Validate the shape of predictions
    if predictions.shape[1] != input_shape[-1]:  # Compare the number of features
        raise ValueError(f"Unexpected predictions shape: {predictions.shape}. Expected (batch_size, {input_shape[-1]}).")

    # Process predictions for output
    num_predictions = predictions.shape[0]
    if predictions.shape[1] > NUM_FEATURES:
        raise ValueError(f"Predictions have more features ({predictions.shape[1]}) than expected NUM_FEATURES ({NUM_FEATURES}).")

    if NUM_FEATURES != predictions.shape[1]:
        dummy_columns = np.random.rand(num_predictions, NUM_FEATURES - predictions.shape[1])
        predictions_with_dummy = np.hstack([predictions, dummy_columns])
        predictions_original_scale = scaler.inverse_transform(predictions_with_dummy)
        predictions_original_scale = predictions_original_scale[:, :predictions.shape[1]]

    else:
        predictions_original_scale = scaler.inverse_transform(predictions)
        predictions_original_scale = predictions_original_scale[:, :predictions.shape[1]]
    

    return predictions_original_scale


In [16]:
def generate_model_prediction_already_scaled(window_packets, scaler, loaded_model, input_shape):
    # Stack the input tensors along a new axis to form a single batch
    window = np.expand_dims(np.stack(window_packets, axis=0), axis=0)

    # Call the model to get predictions
    predictions = loaded_model(window, training=False)

    # Validate the shape of predictions
    if predictions.shape[1] != input_shape[-1]:  # Compare the number of features
        raise ValueError(f"Unexpected predictions shape: {predictions.shape}. Expected (batch_size, {input_shape[-1]}).")

    # Process predictions for output
    num_predictions = predictions.shape[0]
    if predictions.shape[1] > NUM_FEATURES:
        raise ValueError(f"Predictions have more features ({predictions.shape[1]}) than expected NUM_FEATURES ({NUM_FEATURES}).")

    if NUM_FEATURES != predictions.shape[1]:
        dummy_columns = np.random.rand(num_predictions, NUM_FEATURES - predictions.shape[1])
        predictions_with_dummy = np.hstack([predictions, dummy_columns])
        predictions_original_scale = scaler.inverse_transform(predictions_with_dummy)
        predictions_original_scale = predictions_original_scale[:, :predictions.shape[1]]

    else:
        predictions_original_scale = scaler.inverse_transform(predictions)
        predictions_original_scale = predictions_original_scale[:, :predictions.shape[1]]
    

    return predictions_original_scale


### CSV MODE Predictions (two files)

In [ ]:
import pandas as pd
from tqdm import tqdm
from collections import deque
import numpy as np


if MODE == 4:
    # File paths
    #csv_file_path = "data/extracted_csv/DS_Kombat_all_packets.csv"
    csv_file_path = "../../DL/DS_TWO_FILES_SINGLE_MODEL_CG_IPI/modified_packets_mode_4_perc_mod_100_only_DL.csv"
    second_csv_file_path = "../../../FlowSync/Cloud Gaming/DS_CG_IPI/UL/packets_N_to_end_scaled_UL.csv"
    output_file_path = f"modified_packets_mode_{MODE}_perc_mod_{MODIFY_PERCENT}_only_UL.csv"

    # Read the first CSV file
    packets = pd.read_csv(csv_file_path, sep=',', header=0)

    # Open the second CSV file to read packets when needed
    second_file_packets = pd.read_csv(second_csv_file_path, sep=',', header=0)

    window_packets = deque(maxlen=WINDOW_SIZE)  # Sliding window for prediction
    modified_packets = []  # Store modified packets
    second_file_offset = 0  # Start reading from the beginning of the second file

    # Load scalers (example, adjust for your actual function)
    #scaler1_path = '../../FlowSync/Cloud Gaming/DS_CG_PS_IPI/minmax_scaler.joblib'
    #scaler2_path = '../../FlowSync/Cloud Gaming/DS_Other_PS_IPI/minmax_scaler.joblib'
    scaler_remaining_path = '../../../FlowSync/Cloud Gaming/DS_CG_IPI/DL/minmax_scaler_remaining_DL.joblib'

    #scaler_model1, _ = fit_scaler_custom(packets[['IPI']], scaler_path=scaler1_path, power_transformer_path='', load=True, apply_yeo_johnson=False)
    #scaler_model2, _ = fit_scaler_custom(packets[['IPI']], scaler_path=scaler2_path, power_transformer_path='', load=True, apply_yeo_johnson=False)
    scaler_remaining, _ = fit_scaler_custom(packets[['IPI']], scaler_path=scaler_remaining_path,
                                            power_transformer_path='', load=True, apply_yeo_johnson=False)


    # Iterate over packets and modify as needed
    for i, packet in tqdm(packets.iterrows(), total=len(packets), desc="Processing packets", unit="packet"):
        # Make a copy of the packet to ensure original remains unaltered
        modified_packet = packet.copy()


        # Only perform prediction and modification once the window is full
        #if len(window_packets) == WINDOW_SIZE:
        if (i + 1) % WINDOW_SIZE < MODIFY_COUNT:  # Modify condition

            if second_file_offset + WINDOW_SIZE <= len(second_file_packets):
                # Read next WINDOW_SIZE packets from the second file
                next_packets = second_file_packets.iloc[second_file_offset:second_file_offset + WINDOW_SIZE]
                second_file_offset += WINDOW_SIZE  # Move the offset forward by WINDOW_SIZE

                # Add the packets from the second file to the sliding window
                for _, next_packet in next_packets.iterrows():
                    window_packets.append(np.array([next_packet['IPI']], dtype=np.float32))

                #print("window_packets:", window_packets)

            #if (i + 1) % WINDOW_SIZE < MODIFY_COUNT:  # Modify condition
            prediction = None
            if 'Uplink' in packet['Direction']:
                prediction = generate_model_prediction_v1(window_packets, scaler_remaining, loaded_model1, input_shape1)

                # Extract predictions
                predicted_time_offset = prediction[-1][0]  # IPI prediction

                # Apply scaling if needed
                #predicted_time_offset *= 1e6

                # Modify the packet copy with predictions
                modified_packet['IPI'] = predicted_time_offset

        # Append the modified packet to the list
        modified_packets.append(modified_packet)

        # # debugging
        #if i == WINDOW_SIZE * 10:
        #    break

    # Convert the list of modified packets into a DataFrame
    modified_packets_df = pd.DataFrame(modified_packets)

    # Save the modified packets to a new CSV file
    modified_packets_df.to_csv(output_file_path, index=False)
    print(f"Modified packets saved to: {output_file_path}")


Processing packets: 100%|██████████| 393078/393078 [34:39<00:00, 189.01packet/s] 


Modified packets saved to: modified_packets_mode_4_perc_mod_100_only_UL.csv


### Range

In [18]:
packets = pd.read_csv(csv_file_path, sep=',', header=0) # Read the CSV file and specify that the first row is the header

# Retrieve the minimum and maximum values for the IPI and PS fields
min_ipi = packets['IPI'].min()
max_ipi = packets['IPI'].max()

print(min_ipi, max_ipi)

-9.711e-06 0.033329263


In [19]:
print(packets)

                 IPI Direction
0       0.000000e+00    Uplink
1       4.318293e-03    Uplink
2       1.100000e-08  Downlink
3       1.100000e-08  Downlink
4       1.100000e-08  Downlink
...              ...       ...
393073  2.046486e-08  Downlink
393074  2.046486e-08  Downlink
393075  2.046486e-08  Downlink
393076  2.046486e-08  Downlink
393077  2.046486e-08  Downlink

[393078 rows x 2 columns]
